# 데이터 수집

In [1]:
import os

dir_list = ['./dataset/train/', './dataset/test/']

for dir in dir_list:
    if not os.path.exists(dir):
        print(dir)
        os.makedirs(dir)

In [2]:

from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
import random
import shutil

RANDOM_SEED = 42


def find_project_root():
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (candidate / 'captured_images').exists() and (candidate / 'resnet').exists():
            return candidate
    raise FileNotFoundError('captured_images ??? ?? ?????.')


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / 'captured_images'
TRAIN_ROOT = PROJECT_ROOT / 'resnet' / 'dataset' / 'train'
TEST_ROOT = PROJECT_ROOT / 'resnet' / 'dataset' / 'test'


def session_key(file_path):
    parts = file_path.stem.split('_')
    if len(parts) >= 3:
        return f'{parts[1]}_{parts[2]}'
    return file_path.stem


def choose_test_sessions(session_class_counts, class_names, train_ratio=0.8):
    sessions = sorted(session_class_counts)
    total_by_class = Counter()
    for counts in session_class_counts.values():
        total_by_class.update(counts)

    target_test_ratio = 1.0 - train_ratio
    ideal_test_sessions = max(1, round(len(sessions) * target_test_ratio))
    candidate_sizes = sorted({
        max(1, ideal_test_sessions - 1),
        ideal_test_sessions,
        min(len(sessions) - 1, ideal_test_sessions + 1),
    })

    best_score = None
    best_combo = None
    for k in candidate_sizes:
        for combo in combinations(sessions, k):
            test_counts = Counter()
            for sess in combo:
                test_counts.update(session_class_counts[sess])

            missing_penalty = 0
            for cls in class_names:
                if test_counts[cls] == 0:
                    missing_penalty += 1000
                if total_by_class[cls] - test_counts[cls] == 0:
                    missing_penalty += 1000

            ratio_penalty = 0.0
            count_penalty = 0.0
            for cls in class_names:
                target = total_by_class[cls] * target_test_ratio
                count_penalty += abs(test_counts[cls] - target) / max(1, total_by_class[cls])
                ratio_penalty += abs((test_counts[cls] / max(1, total_by_class[cls])) - target_test_ratio)

            size_penalty = abs(k - ideal_test_sessions) * 0.05
            score = missing_penalty + ratio_penalty + count_penalty + size_penalty
            if best_score is None or score < best_score:
                best_score = score
                best_combo = combo

    return set(best_combo)


def dataset_split(src_root=SRC_ROOT, train_ratio=0.8):
    src_root = Path(src_root)
    if not src_root.exists():
        raise FileNotFoundError(f'?? ??? ????: {src_root}')

    random.seed(RANDOM_SEED)
    class_dirs = sorted([p for p in src_root.iterdir() if p.is_dir()])
    class_names = [p.name for p in class_dirs]

    session_class_counts = defaultdict(Counter)
    all_files = []
    for class_dir in class_dirs:
        files = sorted([p for p in class_dir.iterdir() if p.is_file()])
        for file_path in files:
            sess = session_key(file_path)
            session_class_counts[sess][class_dir.name] += 1
            all_files.append((file_path, class_dir.name, sess))

    test_sessions = choose_test_sessions(session_class_counts, class_names, train_ratio=train_ratio)

    for split_root in [TRAIN_ROOT, TEST_ROOT]:
        if split_root.exists():
            shutil.rmtree(split_root)
        split_root.mkdir(parents=True, exist_ok=True)

    split_counts = {'train': Counter(), 'test': Counter()}
    split_sessions = {'train': set(), 'test': set()}

    for file_path, class_name, sess in all_files:
        split_name = 'test' if sess in test_sessions else 'train'
        dst_root = TEST_ROOT if split_name == 'test' else TRAIN_ROOT
        dst_dir = dst_root / class_name
        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file_path, dst_dir / file_path.name)
        split_counts[split_name][class_name] += 1
        split_sessions[split_name].add(sess)

    print('?? ?? split ??')
    print('train:', TRAIN_ROOT)
    print('test:', TEST_ROOT)
    print('train sessions:', sorted(split_sessions['train']))
    print('test sessions:', sorted(split_sessions['test']))
    print('train counts:', dict(split_counts['train']))
    print('test counts:', dict(split_counts['test']))


In [3]:
dataset_split(train_ratio=0.8)

?? ?? split ??
train: C:\Projects\_active\focus_on_class-1_capture\resnet\dataset\train
test: C:\Projects\_active\focus_on_class-1_capture\resnet\dataset\test
train sessions: ['20260423_152308', '20260423_162544', '20260423_162725', '20260423_162748', '20260423_163048', '20260423_163321', '20260423_163345', '20260423_163521', '20260423_163532', '20260423_163709', '20260423_164024', '20260423_164315']
test sessions: ['20260423_162549', '20260423_162950', '20260423_163243']
train counts: {'Attractive': 153, 'Drowsy': 178, 'Inattentive': 148}
test counts: {'Attractive': 37, 'Drowsy': 46, 'Inattentive': 32}


# 데이터로더 만들기

In [4]:

import cv2
import numpy as np
from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

IMAGE_SIZE = 224


class FaceUpperBodyCrop:
    def __init__(self, fallback_crop=(0.10, 0.02, 0.90, 0.92)):
        self.fallback_crop = fallback_crop
        cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        self.face_cascade = cv2.CascadeClassifier(cascade_path)

    def __call__(self, img):
        img = img.convert('RGB')
        arr = np.array(img)
        h, w = arr.shape[:2]
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        faces = self.face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(40, 40))

        if len(faces) > 0:
            x, y, fw, fh = max(faces, key=lambda box: box[2] * box[3])
            cx = x + fw / 2
            x1 = int(cx - fw * 1.5)
            x2 = int(cx + fw * 1.5)
            y1 = int(y - fh * 0.8)
            y2 = int(y + fh * 3.0)
        else:
            left, top, right, bottom = self.fallback_crop
            x1, y1, x2, y2 = int(w * left), int(h * top), int(w * right), int(h * bottom)

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)
        if x2 <= x1 or y2 <= y1:
            return img
        return img.crop((x1, y1, x2, y2))


train_transform = transforms.Compose([
    FaceUpperBodyCrop(),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.03),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08), ratio=(0.3, 3.3)),
])

eval_transform = transforms.Compose([
    FaceUpperBodyCrop(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

tr = train_transform
tr2 = eval_transform


In [5]:
train_dataset = datasets.ImageFolder(str(TRAIN_ROOT), transform=tr)
test_dataset = datasets.ImageFolder(str(TEST_ROOT), transform=tr2)


In [6]:
BATCH_SIZE = 16

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=BATCH_SIZE,
                         shuffle=False)

print('class_names:', train_dataset.classes)
print('train size:', len(train_dataset), 'test size:', len(test_dataset))

class_names: ['Attractive', 'Drowsy', 'Inattentive']
train size: 479 test size: 115


# 모델 준비

In [7]:
from torchvision.models import ResNet34_Weights
from torchvision import models

model = models.resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## 전이학습

In [8]:
# 먼저 전체를 freeze 하고, 뒤쪽 블록만 미세조정합니다.
for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

In [9]:

# ??? ????(? ?? ?? ?? ??? ??)
import torch.nn as nn

model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.fc.in_features, len(train_dataset.classes)),
)
model


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [10]:
# 가중치 변형 금지 여부 확인
for name, module in model.named_parameters():
    print(name, module.requires_grad)

conv1.weight False
bn1.weight False
bn1.bias False
layer1.0.conv1.weight False
layer1.0.bn1.weight False
layer1.0.bn1.bias False
layer1.0.conv2.weight False
layer1.0.bn2.weight False
layer1.0.bn2.bias False
layer1.1.conv1.weight False
layer1.1.bn1.weight False
layer1.1.bn1.bias False
layer1.1.conv2.weight False
layer1.1.bn2.weight False
layer1.1.bn2.bias False
layer1.2.conv1.weight False
layer1.2.bn1.weight False
layer1.2.bn1.bias False
layer1.2.conv2.weight False
layer1.2.bn2.weight False
layer1.2.bn2.bias False
layer2.0.conv1.weight False
layer2.0.bn1.weight False
layer2.0.bn1.bias False
layer2.0.conv2.weight False
layer2.0.bn2.weight False
layer2.0.bn2.bias False
layer2.0.downsample.0.weight False
layer2.0.downsample.1.weight False
layer2.0.downsample.1.bias False
layer2.1.conv1.weight False
layer2.1.bn1.weight False
layer2.1.bn1.bias False
layer2.1.conv2.weight False
layer2.1.bn2.weight False
layer2.1.bn2.bias False
layer2.2.conv1.weight False
layer2.2.bn1.weight False
layer2.2.bn1

# 학습 시키기

In [11]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [12]:

import torch.nn as nn
import torch.optim as optim
from pathlib import Path

model.to(device)  # ?? ?? GPU? ???

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4},
], weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

BEST_CHECKPOINT_PATH = PROJECT_ROOT / 'resnet' / 'resnet34_attention_best.pt'
history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': [],
}
best_test_acc = 0.0


In [13]:
from tqdm import tqdm

epochs = 20

for epoch in range(epochs):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0

    for images, labels in tqdm(train_loader, desc=f'train {epoch + 1}/{epochs}'):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)
        pred = outputs.argmax(dim=1)
        train_correct += (pred == labels).sum().item()

    train_loss = train_loss_sum / len(train_loader.dataset)
    train_acc = train_correct / len(train_loader.dataset)

    model.eval()
    test_loss_sum = 0.0
    test_correct = 0

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f'test  {epoch + 1}/{epochs}'):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            test_loss_sum += loss.item() * images.size(0)
            pred = outputs.argmax(dim=1)
            test_correct += (pred == labels).sum().item()

    test_loss = test_loss_sum / len(test_loader.dataset)
    test_acc = test_correct / len(test_loader.dataset)
    scheduler.step(test_acc)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(f'Epoch {epoch + 1}/{epochs}')
    print(f'  train loss: {train_loss:.4f} | train acc: {train_acc:.4f}')
    print(f'  test  loss: {test_loss:.4f} | test  acc: {test_acc:.4f}')
    print(f'  lr: {optimizer.param_groups[0]["lr"]:.6f}')

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        checkpoint = {
            'model_name': 'resnet34',
            'num_classes': len(train_dataset.classes),
            'class_names': train_dataset.classes,
            'image_size': IMAGE_SIZE,
            'state_dict': model.state_dict(),
            'best_test_acc': best_test_acc,
        }
        torch.save(checkpoint, BEST_CHECKPOINT_PATH)
        print(f'  best model saved -> {BEST_CHECKPOINT_PATH}')

print(f'best test acc: {best_test_acc:.4f}')

test  1/20: 100%|██████████| 8/8 [00:05<00:00,  1.41it/s]


Epoch 1/20
  train loss: 1.0949 | train acc: 0.4447
  test  loss: 1.1528 | test  acc: 0.3913
  lr: 0.000010
  best model saved -> C:\Projects\_active\focus_on_class-1_capture\resnet\resnet34_attention_best.pt


test  2/20: 100%|██████████| 8/8 [00:04<00:00,  1.79it/s]


Epoch 2/20
  train loss: 0.9246 | train acc: 0.5637
  test  loss: 1.1761 | test  acc: 0.3913
  lr: 0.000010


test  3/20: 100%|██████████| 8/8 [00:04<00:00,  1.94it/s]


...

Epoch 18/20
  train loss: 0.5540 | train acc: 0.8100
  test  loss: 1.4339 | test  acc: 0.4000
  lr: 0.000001


test  19/20: 100%|██████████| 8/8 [00:04<00:00,  1.92it/s]


Epoch 19/20
  train loss: 0.5726 | train acc: 0.8017
  test  loss: 1.4704 | test  acc: 0.4000
  lr: 0.000000


test  20/20: 100%|██████████| 8/8 [00:04<00:00,  1.81it/s]

Epoch 20/20
  train loss: 0.5472 | train acc: 0.8455
  test  loss: 1.4962 | test  acc: 0.4000
  lr: 0.000000
best test acc: 0.4174


# 결과확인

# 모델 저장

In [14]:

import torch

CHECKPOINT_PATH = BEST_CHECKPOINT_PATH if 'BEST_CHECKPOINT_PATH' in globals() else PROJECT_ROOT / 'resnet' / 'resnet34_attention_best.pt'

checkpoint = {
    'model_name': 'resnet34',
    'num_classes': len(train_dataset.classes),
    'class_names': train_dataset.classes,
    'image_size': IMAGE_SIZE if 'IMAGE_SIZE' in globals() else 224,
    'state_dict': model.state_dict(),
    'fc_type': 'dropout_linear',
}

torch.save(checkpoint, CHECKPOINT_PATH)
print(f'?? ??: {CHECKPOINT_PATH}')
print(f'??? ??: {train_dataset.classes}')


?? ??: C:\Projects\_active\focus_on_class-1_capture\resnet\resnet34_attention_best.pt
??? ??: ['Attractive', 'Drowsy', 'Inattentive']


# OpenCV 실시간 분류

In [15]:

import time
from collections import deque
import cv2
import torch
from PIL import Image
from torchvision import transforms, models
import torch.nn as nn

CHECKPOINT_PATH = BEST_CHECKPOINT_PATH if 'BEST_CHECKPOINT_PATH' in globals() else PROJECT_ROOT / 'resnet' / 'resnet34_attention_best.pt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
class_names = checkpoint['class_names']
image_size = checkpoint.get('image_size', 224)
state_dict = checkpoint['state_dict']

infer_model = models.resnet34(weights=None)
if 'fc.1.weight' in state_dict:
    infer_model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(infer_model.fc.in_features, len(class_names)),
    )
else:
    infer_model.fc = nn.Linear(infer_model.fc.in_features, len(class_names))
infer_model.load_state_dict(state_dict)
infer_model.to(DEVICE)
infer_model.eval()

transform = transforms.Compose([
    FaceUpperBodyCrop(),
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 960)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 540)

if not cap.isOpened():
    raise RuntimeError('??? ? ? ????.')

INFER_INTERVAL = 3.0  # 3??? 1? ??
SMOOTHING_WINDOW = 5  # ?? N? ?? ???? ?? ??
CONFIDENCE_THRESHOLD = 0.60
prob_history = deque(maxlen=SMOOTHING_WINDOW)

last_infer_time = 0.0
prev_frame_time = time.perf_counter()
pred_label = 'waiting...'
pred_conf = 0.0
top_results = []

while True:
    ret, frame = cap.read()
    if not ret:
        print('???? ?? ?????.')
        break

    frame = cv2.flip(frame, 1)
    now = time.perf_counter()
    fps = 1.0 / max(now - prev_frame_time, 1e-6)
    prev_frame_time = now

    if now - last_infer_time >= INFER_INTERVAL:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(rgb)
        x = transform(pil_img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = infer_model(x)
            probs = torch.softmax(logits[0], dim=0).cpu()

        prob_history.append(probs)
        avg_probs = torch.stack(list(prob_history)).mean(dim=0)
        pred_idx = torch.argmax(avg_probs).item()
        pred_conf = avg_probs[pred_idx].item() * 100
        pred_label = class_names[pred_idx] if avg_probs[pred_idx].item() >= CONFIDENCE_THRESHOLD else 'uncertain'
        top_probs, top_indices = torch.topk(avg_probs, k=min(3, len(class_names)))
        top_results = [
            (class_names[idx.item()], prob.item() * 100)
            for prob, idx in zip(top_probs, top_indices)
        ]
        last_infer_time = now

    remain_sec = max(0.0, INFER_INTERVAL - (now - last_infer_time))

    cv2.rectangle(frame, (10, 10), (500, 190), (0, 0, 0), -1)
    cv2.putText(frame, f'Prediction: {pred_label} ({pred_conf:.1f}%)', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)
    cv2.putText(frame, f'FPS: {fps:.1f}', (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    cv2.putText(frame, f'Smoothing: {len(prob_history)}/{SMOOTHING_WINDOW}', (20, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 220, 255), 1)

    for rank, (label, prob) in enumerate(top_results, start=1):
        text = f'{rank}. {label}: {prob:.1f}%'
        cv2.putText(frame, text, (20, 95 + rank * 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

    cv2.putText(frame, f'Next inference in: {remain_sec:.1f}s', (20, 175), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 220, 255), 1)
    cv2.putText(frame, 'Press q to quit', (310, 175), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
    cv2.imshow('ResNet Real-Time Classification', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
